In [8]:
import pandas as pd
from pathlib import Path
import numpy as np
import json
from sklearn.preprocessing import OneHotEncoder,PowerTransformer
ROOT = Path.cwd().parent
motor='DATA_MODEL_Dev/Motor_data_models.csv'
data_split='DATA_RAW_MODELS/Data_split.json'

# DATA SPLIT

In [9]:
with open(ROOT/data_split, "r", encoding="utf-8") as archivo:
    data_subjects = json.load(archivo)

subjects_train=data_subjects['Train_80']
subjects_test=data_subjects['Test_20']

## Normal Data

In [ ]:
X_train=pd.read_csv(ROOT/motor)
X_train=X_train[X_train['subject_visit'].isin(subjects_train)]
X_train.drop(columns=['subject_visit','UPDRS_III_ProgressionType'], inplace=True)

X_test=pd.read_csv(ROOT/motor)
X_test=X_test[X_test['subject_visit'].isin(subjects_test)]
X_test.drop(columns=['subject_visit','UPDRS_III_ProgressionType'], inplace=True)

y_train=pd.read_csv(ROOT/motor)
y_train=y_train[y_train['subject_visit'].isin(subjects_train)]
y_train=y_train['UPDRS_III_ProgressionType']
y_test=pd.read_csv(ROOT/motor)
y_test=y_test[y_test['subject_visit'].isin(subjects_test)]
y_test=y_test['UPDRS_III_ProgressionType']

## Data Agrupations Stability-Improvement VS Worsening

In [41]:
y_train_2_IS_W= y_train.copy()
y_train_2_IS_W=y_train_2_IS_W.replace({0:0,-1:1,1:1})

## Data Agrupations Stability VS Improvement-Worsening

In [42]:
y_train_2_S_IW= y_train.copy()
y_train_2_S_IW=y_train_2_S_IW.replace({0:0,-1:0,1:1})

## Feature Engineering

In [ ]:
X_train_fe=X_train.copy()
y_train_fe=X_test.copy()


X_train_fe['motor_burden']=X_train_fe['MDS-UPDRS Part III Total Score']+X_train_fe['MDS-UPDRS Part IV Total Score']
X_train_fe['QoL_burden']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']+X_train_fe['MDS-UPDRS Part II Total Score']
X_train_fe['motor_ratio']=X_train_fe['motor_burden']/(X_train_fe['QoL_burden']+1)


X_train_fe['motor_cog_ratio_qol']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_train_fe['MDS-UPDRS Part II Total Score']+1)

X_train_fe['UPDRS1_weight']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)
X_train_fe['UPDRS2_weight']=X_train_fe['MDS-UPDRS Part II Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)
X_train_fe['UPDRS3_weight']=X_train_fe['MDS-UPDRS Part III Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)

X_train_fe['Class3_H&Y']=X_train_fe['UPDRS_III_Class']/(X_train_fe['3.21 HOEHN AND YAHR STAGE']+0.01)
X_train_fe['Functional_impairment']= X_train_fe['SCHWAB & ENGLAND ADL'].apply(lambda x: 100 - x)


encoder = OneHotEncoder(sparse_output=False)

resultados = encoder.fit_transform(X_train_fe[['3.21 HOEHN AND YAHR STAGE']])
columnas_encoder = encoder.get_feature_names_out(['3.21 HOEHN AND YAHR STAGE'])

df_resultados = pd.DataFrame(
    resultados,
    columns=columnas_encoder,
    index=X_train_fe.index
)

X_train_fe = pd.concat([X_train_fe, df_resultados], axis=1)
X_train_fe.drop(columns=['3.21 HOEHN AND YAHR STAGE'], inplace=True)

X_train_fe.head()


,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score,UPDRS_I_Class,...,UPDRS1_weight,UPDRS2_weight,UPDRS3_weight,Class3_H&Y,Functional_impairment,3.21 HOEHN AND YAHR STAGE_0,3.21 HOEHN AND YAHR STAGE_1,3.21 HOEHN AND YAHR STAGE_2,3.21 HOEHN AND YAHR STAGE_3,3.21 HOEHN AND YAHR STAGE_4
0,90,6,5.0,0,35,0.0,0,0.0,46.0,0,...,0.127660,0.106383,0.744681,0.497512,10,0.0,0.0,1.0,0.0,0.0
1,85,9,10.0,0,53,0.0,0,0.0,72.0,0,...,0.123288,0.136986,0.726027,0.332226,15,0.0,0.0,0.0,1.0,0.0
2,80,11,16.0,0,18,7.0,0,0.0,52.0,1,...,0.207547,0.301887,0.339623,0.000000,20,0.0,0.0,1.0,0.0,0.0
3,90,8,8.0,0,34,0.0,0,0.0,50.0,0,...,0.156863,0.156863,0.666667,0.497512,10,0.0,0.0,1.0,0.0,0.0
4,85,4,9.0,0,16,0.0,0,0.0,29.0,0,...,0.133333,0.300000,0.533333,0.000000,15,0.0,0.0,1.0,0.0,0.0


# Feature Selection No Model Univariate

In [18]:
from sklearn.feature_selection import GenericUnivariateSelect
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.feature_selection import VarianceThreshold

In order to perform univariate feature selection we must summarize the methodology and all the methods that we will study:
1. Variance treshold: Fast/Easy it may not capture the all picture. 
2. Using the GenericUnivarianceSelect method we can apply SelectKBest/SelectPercentile and the statistical function associated.

## VarianceThreshold

In [19]:
threshold_range = np.arange(0, 2, 0.1).tolist()

for tresh in threshold_range:
    selector = VarianceThreshold(threshold=tresh)
    X_train_reduced = selector.fit_transform(X_train)
    n_features = X_train_reduced.shape[1]
    print(f'Threshold: {tresh:.2}, Number of features retained: {n_features}')
    # Columnas eliminadas
    eliminadas = X_train.columns[~selector.get_support()]
    print("Deleted:", eliminadas.tolist())

    # Columnas conservadas
    conservadas = X_train.columns[selector.get_support()]
    print("Conserved:", conservadas.tolist())
    print("---------------------------------------------------")



Threshold: 0.0, Number of features retained: 14
Deleted: []
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I (Patient Questionnaire) Total Score', 'MDS-UPDRS Part II Total Score', 'Does participant have DBS', 'MDS-UPDRS Part III Total Score', '3.21 HOEHN AND YAHR STAGE', 'MDS-UPDRS Part IV Total Score', 'DBS_Transition_Visit', 'DBS_Post_Transition', 'MDS-UPDRS Total Score', 'UPDRS_I_Class', 'UPDRS_II_Class', 'UPDRS_III_Class', 'UPDRS_IV_Class']
---------------------------------------------------
Threshold: 0.1, Number of features retained: 10
Deleted: ['Does participant have DBS', 'DBS_Transition_Visit', 'UPDRS_I_Class', 'UPDRS_III_Class']
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I (Patient Questionnaire) Total Score', 'MDS-UPDRS Part II Total Score', 'MDS-UPDRS Part III Total Score', '3.21 HOEHN AND YAHR STAGE', 'MDS-UPDRS Part IV Total Score', 'DBS_Post_Transition', 'MDS-UPDRS Total Score', 'UPDRS_II_Class', 'UPDRS_IV_Class']
------------------------------------------

This method is not usefull due to some features that have low variance are related to not frequent events

## Generic Univariate Select

In [20]:
mode_list = ['percentile', 'k_best']
score_funcs = [f_classif, mutual_info_classif, chi2]
k_list = range(5, 10)
percentile_list = range(10, 100, 10)

results = []

for mode in mode_list:
    for score in score_funcs:

        if mode == 'k_best':
            for k in k_list:
                
                transformer = GenericUnivariateSelect(
                    score_func=score,
                    mode=mode,
                    param=k
                )

                X_new = transformer.fit_transform(X_train, y_train)

                selected_features = X_train.columns[transformer.get_support()]
                delete_features = X_train.columns[~transformer.get_support()]

                results.append({
                    "mode": mode,
                    "score_func": score.__name__,
                    "param": k,
                    "n_features": X_new.shape[1],
                    "features Added": list(selected_features),
                    "features Deleted": list(delete_features)
                })

        elif mode == 'percentile':
            for p in percentile_list:
                
                transformer = GenericUnivariateSelect(
                    score_func=score,
                    mode=mode,
                    param=p
                )

                X_new = transformer.fit_transform(X_train, y_train)

                selected_features = X_train.columns[transformer.get_support()]
                delete_features = X_train.columns[~transformer.get_support()]

                results.append({
                    "mode": mode,
                    "score_func": score.__name__,
                    "param": p,
                    "n_features": X_new.shape[1],
                    "features Added": list(selected_features),
                    "features Deleted": list(delete_features)
                })

df_results = pd.DataFrame(results)
df_results.to_csv(ROOT/'DATA_MODEL_Dev/Motor_feature_selection_results.csv', index=False)
df_results.head()


,mode,score_func,param,n_features,features Added,features Deleted
0,percentile,f_classif,10,2,"[MDS-UPDRS Part III Total Score, MDS-UPDRS Tot...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
1,percentile,f_classif,20,3,"[MDS-UPDRS Part III Total Score, MDS-UPDRS Tot...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
2,percentile,f_classif,30,4,"[MDS-UPDRS Part III Total Score, 3.21 HOEHN AN...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
3,percentile,f_classif,40,6,"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part II Total...",[MDS-UPDRS Part I (Patient Questionnaire) Tota...
4,percentile,f_classif,50,7,"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part II Total...",[MDS-UPDRS Part I (Patient Questionnaire) Tota...


Given that the motor data only present 9 features all will be selected

# Motor Model Development
<p align="center">
    <img src="../../../figures/model_process.png" alt="Model process" width="600">
</p>





---

Linear Models  
(assume a linear decision boundary)

- Logistic Regression  
- Linear Support Vector Machine (Linear SVM)  
- SGD Classifier (logistic loss / hinge loss)

---

Non-Linear Models  
(can model complex decision boundaries)

Tree-Based Models  
- Decision Trees  
- Random Forest  
- Extra Trees (Extremely Randomized Trees)  
- Gradient Boosting  
- XGBoost  
- LightGBM  
- CatBoost  

Distance-Based Models  
- K-Nearest Neighbors (KNN)

Margin-Based Models  
- Support Vector Machines (SVM) with kernels:  
  - RBF kernel  
  - Polynomial kernel  
  - Sigmoid kernel  

Probabilistic Models  
- Multinomial Naive Bayes  
- Bernoulli Naive Bayes  
- Bayesian Classifiers  

Neural Network Models  
- Multilayer Perceptron (MLP)  
---



In [43]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn import svm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import make_scorer, recall_score, confusion_matrix,f1_score,precision_score,balanced_accuracy_score


def specificity_weighted(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]

    total_samples = np.sum(cm)
    specificities = []
    supports = []

    for k in range(n_classes):
        TP = cm[k, k]
        FN = np.sum(cm[k, :]) - TP
        FP = np.sum(cm[:, k]) - TP
        TN = total_samples - (TP + FN + FP)

        spec_k = TN / (TN + FP) if (TN + FP) > 0 else 0
        specificities.append(spec_k)
        supports.append(np.sum(cm[k, :]))

    return np.average(specificities, weights=supports)


def g_mean(y_true, y_pred):
    sens = recall_score(y_true, y_pred, average="macro")
    spec = specificity_weighted(y_true, y_pred)
    return np.sqrt(sens * spec)




def cv_results_to_row(results, modelo_name, parameters, sep=" ± "):

    row = {
        "Model": modelo_name,
        "Parameters": parameters
    }

    for k in results:
        if k.startswith("train_") or k.startswith("test_"):
            prefix, metric = k.split("_", 1)

            mean = results[k].mean()
            std = results[k].std()

            col_name = f"{prefix}_{metric}"
            row[col_name] = f"{mean:.4f}{sep}{std:.4f}"

    return pd.DataFrame([row])


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "acc": "accuracy",
    "bal_acc": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
    "prec_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "rec_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "specificity_weighted": make_scorer(specificity_weighted),
    "g_mean": make_scorer(g_mean),
    
}


## Dummy Classifier

In [44]:
# 1) Dummy model (baseline)
Dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)
# 2) Pipeline
pipe_dummy = Pipeline([
    ("clf", Dummy_model)
])

# 3) Cross-validation
results_dummy = cross_validate(
    pipe_dummy,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# 4) Convertir resultados a fila
df_dummy = cv_results_to_row(
    results_dummy,
    modelo_name="DummyClassifier",
    parameters="Most frequent (Baseline), No Scaler (Pipeline)",
    sep=" ± "
)

# 5) Mostrar resultados
df_dummy


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,DummyClassifier,"Most frequent (Baseline), No Scaler (Pipeline)",0.3956 ± 0.0020,0.3956 ± 0.0005,0.3333 ± 0.0000,0.3333 ± 0.0000,0.1890 ± 0.0007,0.1890 ± 0.0002,0.2243 ± 0.0020,0.2243 ± 0.0005,0.1319 ± 0.0007,0.1319 ± 0.0002,0.3333 ± 0.0000,0.3333 ± 0.0000,0.6044 ± 0.0020,0.6044 ± 0.0005,0.4489 ± 0.0008,0.4488 ± 0.0002


## Linear Models

### Logistic Regression

In [45]:
LogReg_model = LogisticRegression(class_weight="balanced", max_iter=1000, penalty="l2", solver="lbfgs")

#### Normal Data 

##### No Scaler

In [ ]:
# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", LogReg_model)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, No Scaler (Pipeline)",
    sep=" ± "
)

##### StandardScaler

In [46]:
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogReg_model)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_ss,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, StandardScaler (Pipeline)",
    sep=" ± "
)

/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in versi

##### MinMax Scaler

In [47]:
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", LogReg_model)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_mm,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, MinMaxScaler (Pipeline)",
    sep=" ± "
)

/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in versi

In [ ]:



results_none_class2_IS_W = cross_validate(
    pipe_none,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

results_none_class2_S_IW = cross_validate(
    pipe_none,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)



df1_c2_IS_W = cv_results_to_row(
    results_none_class2_IS_W,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, No Scaler (Pipeline), 2 Classes",
    sep=" ± "
)

df1_c2_S_IW = cv_results_to_row(
    results_none_class2_S_IW,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, No Scaler (Pipeline), 2 Classes",
    sep=" ± "
)

# 2) StandardScaler 


results_ss_class2_IS_W = cross_validate(
    pipe_ss,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

results_ss_class2_S_IW = cross_validate(
    pipe_ss,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)


df2_c2_IS_W = cv_results_to_row(
    results_ss_class2_IS_W,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, StandardScaler (Pipeline), 2 Classes",
    sep=" ± "
)

df2_c2_S_IW = cv_results_to_row(
    results_ss_class2_S_IW,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, StandardScaler (Pipeline), 2 Classes",
    sep=" ± "
)



results_mm_class2_IS_W = cross_validate(
    pipe_mm,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

results_mm_class2_S_IW = cross_validate(
    pipe_mm,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)



df3_c2_IS_W = cv_results_to_row(
    results_mm_class2_IS_W,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, MinMaxScaler (Pipeline), 2 Classes",
    sep=" ± "
)

df3_c2_S_IW = cv_results_to_row(
    results_mm_class2_S_IW,
    modelo_name="LogisticRegression",
    parameters="Balanced, max_iter=1000, Default Model, MinMaxScaler (Pipeline), 2 Classes",
    sep=" ± "
)


final_results = pd.concat([df1, df2, df3, df1_c2_IS_W, df2_c2_IS_W, df3_c2_IS_W, df1_c2_S_IW, df2_c2_S_IW, df3_c2_S_IW], ignore_index=True)
final_results


/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,"Balanced, max_iter=1000, Default Model, No Sca...",0.4410 ± 0.0559,0.4828 ± 0.0190,0.4308 ± 0.0579,0.4787 ± 0.0195,0.4288 ± 0.0611,0.4754 ± 0.0196,0.4419 ± 0.0586,0.4842 ± 0.0194,0.4342 ± 0.0632,0.4756 ± 0.0194,0.4308 ± 0.0579,0.4787 ± 0.0195,0.7163 ± 0.0273,0.7377 ± 0.0097,0.5548 ± 0.0478,0.5942 ± 0.0160
1,LogisticRegression,"Balanced, max_iter=1000, Default Model, Standa...",0.4314 ± 0.0536,0.4780 ± 0.0177,0.4198 ± 0.0548,0.4742 ± 0.0180,0.4181 ± 0.0581,0.4707 ± 0.0181,0.4321 ± 0.0557,0.4793 ± 0.0180,0.4237 ± 0.0597,0.4710 ± 0.0180,0.4198 ± 0.0548,0.4742 ± 0.0180,0.7109 ± 0.0249,0.7354 ± 0.0093,0.5456 ± 0.0448,0.5905 ± 0.0149
2,LogisticRegression,"Balanced, max_iter=1000, Default Model, MinMax...",0.4533 ± 0.0563,0.4828 ± 0.0159,0.4369 ± 0.0577,0.4729 ± 0.0168,0.4355 ± 0.0610,0.4719 ± 0.0171,0.4520 ± 0.0589,0.4831 ± 0.0164,0.4393 ± 0.0637,0.4715 ± 0.0171,0.4369 ± 0.0577,0.4729 ± 0.0168,0.7163 ± 0.0287,0.7312 ± 0.0085,0.5588 ± 0.0483,0.5880 ± 0.0139
3,LogisticRegression,"Balanced, max_iter=1000, Default Model, No Sca...",0.5701 ± 0.0368,0.6113 ± 0.0195,0.5713 ± 0.0430,0.6147 ± 0.0183,0.5635 ± 0.0403,0.6063 ± 0.0188,0.5740 ± 0.0376,0.6155 ± 0.0193,0.5683 ± 0.0411,0.6098 ± 0.0175,0.5713 ± 0.0430,0.6147 ± 0.0183,0.5724 ± 0.0502,0.6181 ± 0.0173,0.5718 ± 0.0465,0.6164 ± 0.0177
4,LogisticRegression,"Balanced, max_iter=1000, Default Model, Standa...",0.5660 ± 0.0454,0.6123 ± 0.0139,0.5673 ± 0.0496,0.6148 ± 0.0123,0.5594 ± 0.0472,0.6069 ± 0.0131,0.5699 ± 0.0454,0.6165 ± 0.0137,0.5647 ± 0.0476,0.6099 ± 0.0118,0.5673 ± 0.0496,0.6148 ± 0.0123,0.5685 ± 0.0553,0.6173 ± 0.0111,0.5678 ± 0.0523,0.6160 ± 0.0117
5,LogisticRegression,"Balanced, max_iter=1000, Default Model, MinMax...",0.5729 ± 0.0340,0.6126 ± 0.0157,0.5711 ± 0.0411,0.6121 ± 0.0123,0.5645 ± 0.0377,0.6057 ± 0.0137,0.5763 ± 0.0345,0.6164 ± 0.0150,0.5685 ± 0.0396,0.6079 ± 0.0125,0.5711 ± 0.0411,0.6121 ± 0.0123,0.5694 ± 0.0497,0.6115 ± 0.0108,0.5702 ± 0.0453,0.6118 ± 0.0112
6,LogisticRegression,"Balanced, max_iter=1000, Default Model, No Sca...",0.5810 ± 0.0467,0.6202 ± 0.0131,0.5613 ± 0.0503,0.6101 ± 0.0160,0.5321 ± 0.0442,0.5743 ± 0.0138,0.6080 ± 0.0423,0.6449 ± 0.0122,0.5471 ± 0.0381,0.5839 ± 0.0123,0.5613 ± 0.0503,0.6101 ± 0.0160,0.5416 ± 0.0609,0.6001 ± 0.0195,0.5513 ± 0.0549,0.6051 ± 0.0177
7,LogisticRegression,"Balanced, max_iter=1000, Default Model, Standa...",0.5797 ± 0.0340,0.6209 ± 0.0134,0.5511 ± 0.0438,0.6101 ± 0.0163,0.5256 ± 0.0350,0.5746 ± 0.0140,0.6060 ± 0.0307,0.6455 ± 0.0124,0.5391 ± 0.0328,0.5840 ± 0.0125,0.5511 ± 0.0438,0.6101 ± 0.0163,0.5225 ± 0.0629,0.5994 ± 0.0202,0.5364 ± 0.0530,0.6047 ± 0.0182
8,LogisticRegression,"Balanced, max_iter=1000, Default Model, MinMax...",0.5837 ± 0.0455,0.6212 ± 0.0127,0.5762 ± 0.0568,0.6169 ± 0.0161,0.5408 ± 0.0480,0.5779 ± 0.0135,0.6117 ± 0.0422,0.6462 ± 0.0118,0.5575 ± 0.0429,0.5887 ± 0.0122,0.5762 ± 0.0568,0.6169 ± 0.0161,0.5686 ± 0.0689,0.6126 ± 0.0207,0.5723 ± 0.0628,0.6147 ± 0.0183


### Logistic Regression SFS_Forward

In [16]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(LogReg_model),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(LogReg_model)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="LogReg SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(LogReg_model)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="LogReg SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(LogReg_model)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="LogReg SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_logreg_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_logreg_sfs_forward

/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/fsc/Desktop/PD_Proj

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogReg SFS,Forward selection (n_features=10),0.4396 ± 0.0675,0.4784 ± 0.0226,0.4315 ± 0.0728,0.4733 ± 0.0214,0.4269 ± 0.0750,0.4694 ± 0.0221,0.4392 ± 0.0719,0.4792 ± 0.0223,0.4326 ± 0.0790,0.4708 ± 0.0212,0.4315 ± 0.0728,0.4733 ± 0.0214,0.7172 ± 0.0398,0.7367 ± 0.0094,0.5553 ± 0.0628,0.5904 ± 0.0170
1,LogReg SFS,"Forward selection (n_features=10), StandardScaler",0.4423 ± 0.0461,0.4839 ± 0.0208,0.4336 ± 0.0503,0.4800 ± 0.0210,0.4304 ± 0.0522,0.4760 ± 0.0212,0.4425 ± 0.0497,0.4853 ± 0.0208,0.4362 ± 0.0554,0.4774 ± 0.0206,0.4336 ± 0.0503,0.4800 ± 0.0210,0.7169 ± 0.0296,0.7406 ± 0.0100,0.5571 ± 0.0434,0.5961 ± 0.0170
2,LogReg SFS,"Forward selection (n_features=10), MinMaxScaler",0.4465 ± 0.0471,0.4873 ± 0.0176,0.4349 ± 0.0481,0.4777 ± 0.0186,0.4334 ± 0.0505,0.4758 ± 0.0188,0.4471 ± 0.0494,0.4871 ± 0.0178,0.4379 ± 0.0528,0.4761 ± 0.0189,0.4349 ± 0.0481,0.4777 ± 0.0186,0.7165 ± 0.0251,0.7349 ± 0.0088,0.5577 ± 0.0407,0.5925 ± 0.0150


### Logistic Regression Feature Enginering

In [24]:

# 1) NO SCALER 
pipe_yj_none = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("clf", LogReg_model)
])

results_none = cross_validate(
    pipe_yj_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Logistic_Regression FE",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_yj_ss = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler()),
    ("clf", LogReg_model)
])


results_ss = cross_validate(
    pipe_yj_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Logistic_Regression FE",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_yj_mm = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("scaler", MinMaxScaler()),
    ("clf", LogReg_model)
])

results_mm = cross_validate(
    pipe_yj_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Logistic_Regression FE",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_lr_fe = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_lr_fe

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Logistic_Regression FE,"Default, No Scaler (Pipeline)",0.4464 ± 0.0534,0.4900 ± 0.0143,0.4358 ± 0.0534,0.4845 ± 0.0156,0.4324 ± 0.0564,0.4809 ± 0.0153,0.4454 ± 0.0561,0.4901 ± 0.0146,0.4368 ± 0.0590,0.4814 ± 0.0152,0.4358 ± 0.0534,0.4845 ± 0.0156,0.7178 ± 0.0277,0.7396 ± 0.0082,0.5588 ± 0.0448,0.5986 ± 0.0130
1,Logistic_Regression FE,"Default, StandardScaler (Pipeline)",0.4464 ± 0.0534,0.4900 ± 0.0143,0.4358 ± 0.0534,0.4845 ± 0.0156,0.4324 ± 0.0564,0.4809 ± 0.0153,0.4454 ± 0.0561,0.4901 ± 0.0146,0.4368 ± 0.0590,0.4814 ± 0.0152,0.4358 ± 0.0534,0.4845 ± 0.0156,0.7178 ± 0.0277,0.7396 ± 0.0082,0.5588 ± 0.0448,0.5986 ± 0.0130
2,Logistic_Regression FE,"Default, MinMaxScaler (Pipeline)",0.4409 ± 0.0437,0.4856 ± 0.0198,0.4275 ± 0.0483,0.4787 ± 0.0228,0.4247 ± 0.0502,0.4760 ± 0.0226,0.4400 ± 0.0475,0.4856 ± 0.0210,0.4280 ± 0.0520,0.4758 ± 0.0224,0.4275 ± 0.0483,0.4787 ± 0.0228,0.7126 ± 0.0287,0.7353 ± 0.0124,0.5515 ± 0.0418,0.5932 ± 0.0191


### Linear Support Vector Machine (Linear SVM)

In [17]:

# Modelo base
linear_svc = svm.SVC(kernel="linear")

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", linear_svc)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SVC_Linear",
    parameters="Default Model, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", linear_svc)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SVC_Linear",
    parameters="Default Model, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", linear_svc)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SVC_Linear",
    parameters="Default Model, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_svc = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_svc


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVC_Linear,"Default Model, No Scaler (Pipeline)",0.4602 ± 0.0347,0.4924 ± 0.0121,0.4091 ± 0.0302,0.4401 ± 0.0145,0.3672 ± 0.0318,0.4006 ± 0.0250,0.4080 ± 0.0324,0.4407 ± 0.0206,0.3686 ± 0.0650,0.4920 ± 0.0086,0.4091 ± 0.0302,0.4401 ± 0.0145,0.6738 ± 0.0185,0.6906 ± 0.0109,0.5249 ± 0.0264,0.5513 ± 0.0135
1,SVC_Linear,"Default Model, StandardScaler (Pipeline)",0.4574 ± 0.0374,0.4921 ± 0.0102,0.4062 ± 0.0316,0.4397 ± 0.0132,0.3628 ± 0.0293,0.3993 ± 0.0240,0.4041 ± 0.0318,0.4399 ± 0.0192,0.3610 ± 0.0555,0.4731 ± 0.0162,0.4062 ± 0.0316,0.4397 ± 0.0132,0.6719 ± 0.0196,0.6910 ± 0.0101,0.5223 ± 0.0278,0.5512 ± 0.0123
2,SVC_Linear,"Default Model, MinMaxScaler (Pipeline)",0.4478 ± 0.0368,0.4863 ± 0.0060,0.3956 ± 0.0319,0.4321 ± 0.0099,0.3508 ± 0.0330,0.3913 ± 0.0221,0.3914 ± 0.0341,0.4317 ± 0.0169,0.3638 ± 0.0764,0.4632 ± 0.0298,0.3956 ± 0.0319,0.4321 ± 0.0099,0.6633 ± 0.0187,0.6843 ± 0.0083,0.5121 ± 0.0276,0.5438 ± 0.0094


###  Linear SVM SFS_Forward

In [18]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(linear_svc),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(linear_svc)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="LinearSVC SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(linear_svc)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="LinearSVC SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(linear_svc)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="LinearSVC SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_linear_svc_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_linear_svc_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LinearSVC SFS,Forward selection (n_features=10),0.4602 ± 0.0284,0.4856 ± 0.0091,0.4081 ± 0.0222,0.4333 ± 0.0105,0.3637 ± 0.0187,0.3943 ± 0.0195,0.4052 ± 0.0212,0.4340 ± 0.0154,0.3659 ± 0.0473,0.4813 ± 0.0234,0.4081 ± 0.0222,0.4333 ± 0.0105,0.6722 ± 0.0142,0.6855 ± 0.0078,0.5237 ± 0.0197,0.5450 ± 0.0097
1,LinearSVC SFS,"Forward selection (n_features=10), StandardScaler",0.4464 ± 0.0215,0.4876 ± 0.0089,0.3949 ± 0.0143,0.4341 ± 0.0129,0.3481 ± 0.0073,0.3916 ± 0.0249,0.3903 ± 0.0092,0.4327 ± 0.0200,0.3270 ± 0.0248,0.4992 ± 0.0343,0.3949 ± 0.0143,0.4341 ± 0.0129,0.6654 ± 0.0062,0.6859 ± 0.0106,0.5125 ± 0.0114,0.5456 ± 0.0123
2,LinearSVC SFS,"Forward selection (n_features=10), MinMaxScaler",0.4547 ± 0.0304,0.4900 ± 0.0143,0.4026 ± 0.0260,0.4353 ± 0.0179,0.3578 ± 0.0281,0.3895 ± 0.0327,0.3990 ± 0.0274,0.4320 ± 0.0265,0.3630 ± 0.0819,0.4424 ± 0.0960,0.4026 ± 0.0260,0.4353 ± 0.0179,0.6669 ± 0.0154,0.6862 ± 0.0127,0.5180 ± 0.0224,0.5465 ± 0.0163


### SGD Classifier 

In [19]:

# Modelo base
SGD_classifier = SGDClassifier(loss="hinge")

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", SGD_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_Classifier",
    parameters="Default Model, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SGD_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SGD_Classifier",
    parameters="Default Model, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", SGD_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SGD_Classifier",
    parameters="Default Model, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_sgd = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_sgd

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_Classifier,"Default Model, No Scaler (Pipeline)",0.3957 ± 0.0208,0.4062 ± 0.0076,0.3735 ± 0.0246,0.3872 ± 0.0344,0.2650 ± 0.0460,0.2706 ± 0.0473,0.2756 ± 0.0380,0.2800 ± 0.0292,0.2793 ± 0.1255,0.2889 ± 0.1101,0.3735 ± 0.0246,0.3872 ± 0.0344,0.6687 ± 0.0355,0.6749 ± 0.0414,0.4997 ± 0.0281,0.5112 ± 0.0383
1,SGD_Classifier,"Default Model, StandardScaler (Pipeline)",0.4067 ± 0.0278,0.4224 ± 0.0559,0.3985 ± 0.0423,0.4095 ± 0.0301,0.3843 ± 0.0394,0.3940 ± 0.0406,0.3982 ± 0.0343,0.4085 ± 0.0537,0.4062 ± 0.0613,0.4163 ± 0.0279,0.3985 ± 0.0423,0.4095 ± 0.0301,0.6970 ± 0.0446,0.7035 ± 0.0142,0.5268 ± 0.0443,0.5363 ± 0.0190
2,SGD_Classifier,"Default Model, MinMaxScaler (Pipeline)",0.4163 ± 0.0312,0.4440 ± 0.0540,0.3901 ± 0.0508,0.4164 ± 0.0390,0.3263 ± 0.0721,0.3584 ± 0.0753,0.3505 ± 0.0614,0.3816 ± 0.0732,0.3997 ± 0.1249,0.5214 ± 0.0497,0.3901 ± 0.0508,0.4164 ± 0.0390,0.6739 ± 0.0628,0.6910 ± 0.0446,0.5126 ± 0.0565,0.5360 ± 0.0370


### SGD Classifier SFS_Forward

In [20]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(SGD_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(SGD_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="SGD Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SGD_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="SGD Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SGD_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="SGD Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_sgd_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_sgd_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD Classifier SFS,Forward selection (n_features=10),0.4574 ± 0.0373,0.4485 ± 0.0306,0.4214 ± 0.0390,0.4167 ± 0.0466,0.3583 ± 0.0746,0.3510 ± 0.0845,0.3869 ± 0.0690,0.3762 ± 0.0773,0.4106 ± 0.0613,0.4672 ± 0.0880,0.4214 ± 0.0390,0.4167 ± 0.0466,0.6887 ± 0.0414,0.6860 ± 0.0501,0.5386 ± 0.0409,0.5345 ± 0.0492
1,SGD Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.3955 ± 0.0494,0.4015 ± 0.0406,0.3894 ± 0.0444,0.3946 ± 0.0258,0.3733 ± 0.0535,0.3843 ± 0.0281,0.3838 ± 0.0565,0.3958 ± 0.0363,0.3857 ± 0.0558,0.3998 ± 0.0271,0.3894 ± 0.0444,0.3946 ± 0.0258,0.6896 ± 0.0295,0.7000 ± 0.0082,0.5178 ± 0.0394,0.5253 ± 0.0179
2,SGD Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4574 ± 0.0267,0.4688 ± 0.0089,0.4199 ± 0.0279,0.4324 ± 0.0103,0.3743 ± 0.0454,0.3889 ± 0.0350,0.4055 ± 0.0430,0.4186 ± 0.0323,0.4361 ± 0.0619,0.4533 ± 0.0812,0.4199 ± 0.0279,0.4324 ± 0.0103,0.6916 ± 0.0208,0.6996 ± 0.0082,0.5388 ± 0.0260,0.5500 ± 0.0096


## Non-Linear Models  

### Tree-Base Models

#### Decision Tree

In [21]:

# Modelo base
Tree_classifier = DecisionTreeClassifier(max_depth=5,
                                         min_samples_leaf=10,
                                         min_samples_split=20, random_state=42)

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", Tree_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Decision_Tree_Classifier",
    parameters="Gini, max_depth=5, min_samples_leaf=10, min_samples_split=20, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", Tree_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Decision_Tree_Classifier",
    parameters="Gini, max_depth=5, min_samples_leaf=10, min_samples_split=20, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", Tree_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Decision_Tree_Classifier",
    parameters="Gini, max_depth=5, min_samples_leaf=10, min_samples_split=20, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_tree = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_tree

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Decision_Tree_Classifier,"Gini, max_depth=5, min_samples_leaf=10, min_sa...",0.4245 ± 0.0486,0.5505 ± 0.0282,0.3959 ± 0.0433,0.5256 ± 0.0296,0.3880 ± 0.0424,0.5197 ± 0.0328,0.4091 ± 0.0449,0.5352 ± 0.0320,0.4218 ± 0.0541,0.5704 ± 0.0214,0.3959 ± 0.0433,0.5256 ± 0.0296,0.6749 ± 0.0284,0.7460 ± 0.0217,0.5165 ± 0.0385,0.6262 ± 0.0265
1,Decision_Tree_Classifier,"Gini, max_depth=5, min_samples_leaf=10, min_sa...",0.4245 ± 0.0486,0.5505 ± 0.0282,0.3959 ± 0.0433,0.5256 ± 0.0296,0.3880 ± 0.0424,0.5197 ± 0.0328,0.4091 ± 0.0449,0.5352 ± 0.0320,0.4218 ± 0.0541,0.5704 ± 0.0214,0.3959 ± 0.0433,0.5256 ± 0.0296,0.6749 ± 0.0284,0.7460 ± 0.0217,0.5165 ± 0.0385,0.6262 ± 0.0265
2,Decision_Tree_Classifier,"Gini, max_depth=5, min_samples_leaf=10, min_sa...",0.4245 ± 0.0486,0.5505 ± 0.0282,0.3959 ± 0.0433,0.5256 ± 0.0296,0.3880 ± 0.0424,0.5197 ± 0.0328,0.4091 ± 0.0449,0.5352 ± 0.0320,0.4218 ± 0.0541,0.5704 ± 0.0214,0.3959 ± 0.0433,0.5256 ± 0.0296,0.6749 ± 0.0284,0.7460 ± 0.0217,0.5165 ± 0.0385,0.6262 ± 0.0265


#### Tree-Base Model SFS_Forward

In [22]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(Tree_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(Tree_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="Tree Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(Tree_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="Tree Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(Tree_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="Tree Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_tree_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_tree_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree Classifier SFS,forward selection (n_features=10),0.4437 ± 0.0309,0.5450 ± 0.0183,0.4142 ± 0.0282,0.5159 ± 0.0167,0.4059 ± 0.0307,0.5103 ± 0.0197,0.4281 ± 0.0327,0.5291 ± 0.0191,0.4308 ± 0.0288,0.5538 ± 0.0240,0.4142 ± 0.0282,0.5159 ± 0.0167,0.6873 ± 0.0233,0.7426 ± 0.0122,0.5335 ± 0.0265,0.6189 ± 0.0148
1,Tree Classifier SFS,"forward selection (n_features=10), StandardScaler",0.4437 ± 0.0309,0.5450 ± 0.0183,0.4142 ± 0.0282,0.5159 ± 0.0167,0.4059 ± 0.0307,0.5103 ± 0.0197,0.4281 ± 0.0327,0.5291 ± 0.0191,0.4308 ± 0.0288,0.5538 ± 0.0240,0.4142 ± 0.0282,0.5159 ± 0.0167,0.6873 ± 0.0233,0.7426 ± 0.0122,0.5335 ± 0.0265,0.6189 ± 0.0148
2,Tree Classifier SFS,"forward selection (n_features=10), MinMaxScaler",0.4437 ± 0.0309,0.5450 ± 0.0183,0.4142 ± 0.0282,0.5159 ± 0.0167,0.4059 ± 0.0307,0.5103 ± 0.0197,0.4281 ± 0.0327,0.5291 ± 0.0191,0.4308 ± 0.0288,0.5538 ± 0.0240,0.4142 ± 0.0282,0.5159 ± 0.0167,0.6873 ± 0.0233,0.7426 ± 0.0122,0.5335 ± 0.0265,0.6189 ± 0.0148


#### Random Forest

In [23]:

# Modelo base
RF_classifier = RandomForestClassifier(
    n_estimators=1000,
    max_depth=10,           
    min_samples_leaf=5,     
    bootstrap=True,
    random_state=42,
)


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", RF_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Random_Forest_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RF_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Random_Forest_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", RF_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Random_Forest_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_rf = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_rf

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Random_Forest_Classifier,"Default, No Scaler (Pipeline)",0.4340 ± 0.0384,0.6913 ± 0.0059,0.3961 ± 0.0333,0.6521 ± 0.0042,0.3775 ± 0.0344,0.6548 ± 0.0046,0.4047 ± 0.0354,0.6744 ± 0.0035,0.4281 ± 0.0707,0.7399 ± 0.0175,0.3961 ± 0.0333,0.6521 ± 0.0042,0.6680 ± 0.0186,0.8149 ± 0.0036,0.5142 ± 0.0284,0.7289 ± 0.0039
1,Random_Forest_Classifier,"Default, StandardScaler (Pipeline)",0.4354 ± 0.0388,0.6923 ± 0.0049,0.3966 ± 0.0350,0.6531 ± 0.0030,0.3769 ± 0.0376,0.6560 ± 0.0040,0.4049 ± 0.0373,0.6755 ± 0.0023,0.4278 ± 0.0735,0.7408 ± 0.0170,0.3966 ± 0.0350,0.6531 ± 0.0030,0.6682 ± 0.0192,0.8155 ± 0.0030,0.5145 ± 0.0298,0.7298 ± 0.0030
2,Random_Forest_Classifier,"Default, MinMaxScaler (Pipeline)",0.4340 ± 0.0384,0.6913 ± 0.0056,0.3961 ± 0.0333,0.6521 ± 0.0041,0.3775 ± 0.0344,0.6548 ± 0.0047,0.4047 ± 0.0354,0.6744 ± 0.0034,0.4281 ± 0.0707,0.7392 ± 0.0180,0.3961 ± 0.0333,0.6521 ± 0.0041,0.6680 ± 0.0186,0.8150 ± 0.0035,0.5142 ± 0.0284,0.7290 ± 0.0038


#### Random Forest SFS_Fackward

In [24]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# Forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(RF_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(RF_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="RF Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(RF_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="RF Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(RF_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="RF Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_rf_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_rf_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF Classifier SFS,forward selection (n_features=10),0.4259 ± 0.0324,0.6360 ± 0.0219,0.3868 ± 0.0353,0.5975 ± 0.0221,0.3639 ± 0.0445,0.5968 ± 0.0230,0.3943 ± 0.0391,0.6179 ± 0.0226,0.3830 ± 0.0505,0.6660 ± 0.0202,0.3868 ± 0.0353,0.5975 ± 0.0221,0.6665 ± 0.0224,0.7846 ± 0.0141,0.5075 ± 0.0313,0.6847 ± 0.0188
1,RF Classifier SFS,"forward selection (n_features=10), StandardScaler",0.4273 ± 0.0314,0.6398 ± 0.0134,0.3886 ± 0.0356,0.6025 ± 0.0162,0.3684 ± 0.0442,0.6031 ± 0.0191,0.3979 ± 0.0383,0.6231 ± 0.0168,0.3976 ± 0.0567,0.6698 ± 0.0075,0.3886 ± 0.0356,0.6025 ± 0.0162,0.6669 ± 0.0225,0.7868 ± 0.0101,0.5088 ± 0.0312,0.6885 ± 0.0136
2,RF Classifier SFS,"forward selection (n_features=10), MinMaxScaler",0.4231 ± 0.0317,0.6374 ± 0.0209,0.3846 ± 0.0353,0.5981 ± 0.0217,0.3628 ± 0.0449,0.5971 ± 0.0228,0.3928 ± 0.0395,0.6187 ± 0.0222,0.3794 ± 0.0497,0.6676 ± 0.0173,0.3846 ± 0.0353,0.5981 ± 0.0217,0.6653 ± 0.0231,0.7851 ± 0.0139,0.5056 ± 0.0316,0.6852 ± 0.0184


#### Extra Trees 

In [25]:

# Modelo base
extratree_classifier = ExtraTreesClassifier(max_depth=10)


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", extratree_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Extra_Trees_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", extratree_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Extra_Trees_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", extratree_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Extra_Trees_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_et = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_et

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Extra_Trees_Classifier,"Default, No Scaler (Pipeline)",0.4396 ± 0.0186,0.7950 ± 0.0153,0.3957 ± 0.0159,0.7636 ± 0.0179,0.3700 ± 0.0148,0.7796 ± 0.0179,0.4018 ± 0.0121,0.7891 ± 0.0165,0.4088 ± 0.0387,0.8543 ± 0.0087,0.3957 ± 0.0159,0.7636 ± 0.0179,0.6677 ± 0.0038,0.8705 ± 0.0100,0.5140 ± 0.0116,0.8153 ± 0.0142
1,Extra_Trees_Classifier,"Default, StandardScaler (Pipeline)",0.4299 ± 0.0166,0.7984 ± 0.0141,0.3872 ± 0.0111,0.7684 ± 0.0161,0.3615 ± 0.0067,0.7842 ± 0.0165,0.3923 ± 0.0081,0.7929 ± 0.0153,0.4024 ± 0.0222,0.8536 ± 0.0104,0.3872 ± 0.0111,0.7684 ± 0.0161,0.6607 ± 0.0047,0.8733 ± 0.0087,0.5058 ± 0.0088,0.8192 ± 0.0126
2,Extra_Trees_Classifier,"Default, MinMaxScaler (Pipeline)",0.4285 ± 0.0237,0.8005 ± 0.0118,0.3848 ± 0.0213,0.7707 ± 0.0129,0.3567 ± 0.0211,0.7869 ± 0.0122,0.3891 ± 0.0215,0.7954 ± 0.0121,0.3840 ± 0.0348,0.8560 ± 0.0063,0.3848 ± 0.0213,0.7707 ± 0.0129,0.6611 ± 0.0130,0.8744 ± 0.0081,0.5043 ± 0.0187,0.8209 ± 0.0107


#### Extra Trees SFS_Forward

In [26]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(extratree_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(extratree_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="ExtraTree Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(extratree_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="ExtraTree Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(extratree_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="ExtraTree Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_extratree_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_extratree_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,ExtraTree Classifier SFS,forward selection (n_features=10),0.4231 ± 0.0491,0.7734 ± 0.0315,0.3834 ± 0.0401,0.7436 ± 0.0341,0.3593 ± 0.0342,0.7576 ± 0.0337,0.3895 ± 0.0407,0.7670 ± 0.0328,0.3909 ± 0.0474,0.8296 ± 0.0187,0.3834 ± 0.0401,0.7436 ± 0.0341,0.6632 ± 0.0239,0.8606 ± 0.0210,0.5039 ± 0.0355,0.7999 ± 0.0281
1,ExtraTree Classifier SFS,"forward selection (n_features=10), StandardScaler",0.4272 ± 0.0311,0.7847 ± 0.0101,0.3873 ± 0.0244,0.7535 ± 0.0114,0.3664 ± 0.0199,0.7686 ± 0.0119,0.3961 ± 0.0231,0.7783 ± 0.0109,0.3850 ± 0.0299,0.8425 ± 0.0085,0.3873 ± 0.0244,0.7535 ± 0.0114,0.6649 ± 0.0164,0.8654 ± 0.0063,0.5074 ± 0.0219,0.8075 ± 0.0091
2,ExtraTree Classifier SFS,"forward selection (n_features=10), MinMaxScaler",0.4272 ± 0.0317,0.8012 ± 0.0148,0.3905 ± 0.0318,0.7727 ± 0.0168,0.3733 ± 0.0349,0.7875 ± 0.0164,0.4015 ± 0.0333,0.7960 ± 0.0156,0.3918 ± 0.0422,0.8496 ± 0.0084,0.3905 ± 0.0318,0.7727 ± 0.0168,0.6719 ± 0.0206,0.8768 ± 0.0100,0.5120 ± 0.0287,0.8231 ± 0.0136


#### Gradient Boosting

In [27]:

# Modelo base
GB_classifier = GradientBoostingClassifier()


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", GB_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="Gradient_Boosting_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GB_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="Gradient_Boosting_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", GB_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="Gradient_Boosting_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_gb = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_gb

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Gradient_Boosting_Classifier,"Default, No Scaler (Pipeline)",0.4341 ± 0.0262,0.7703 ± 0.0169,0.4076 ± 0.0250,0.7484 ± 0.0193,0.4011 ± 0.0279,0.7593 ± 0.0191,0.4202 ± 0.0255,0.7667 ± 0.0177,0.4185 ± 0.0401,0.7933 ± 0.0148,0.4076 ± 0.0250,0.7484 ± 0.0193,0.6821 ± 0.0089,0.8650 ± 0.0103,0.5271 ± 0.0193,0.8046 ± 0.0151
1,Gradient_Boosting_Classifier,"Default, StandardScaler (Pipeline)",0.4382 ± 0.0349,0.7703 ± 0.0166,0.4112 ± 0.0353,0.7488 ± 0.0190,0.4050 ± 0.0396,0.7599 ± 0.0189,0.4244 ± 0.0358,0.7669 ± 0.0175,0.4241 ± 0.0540,0.7940 ± 0.0148,0.4112 ± 0.0353,0.7488 ± 0.0190,0.6839 ± 0.0132,0.8649 ± 0.0102,0.5300 ± 0.0278,0.8047 ± 0.0149
2,Gradient_Boosting_Classifier,"Default, MinMaxScaler (Pipeline)",0.4355 ± 0.0301,0.7723 ± 0.0165,0.4086 ± 0.0274,0.7510 ± 0.0188,0.4018 ± 0.0293,0.7619 ± 0.0188,0.4212 ± 0.0286,0.7690 ± 0.0173,0.4192 ± 0.0411,0.7945 ± 0.0148,0.4086 ± 0.0274,0.7510 ± 0.0188,0.6829 ± 0.0120,0.8665 ± 0.0101,0.5280 ± 0.0219,0.8067 ± 0.0148


#### Gradient Boosting SFS_Forward

In [28]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(GB_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(GB_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="Gradient Boosting Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(GB_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="Gradient Boosting Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(GB_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="Gradient Boosting Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_gradient_boosting_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_gradient_boosting_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Gradient Boosting Classifier SFS,forward selection (n_features=10),0.4396 ± 0.0409,0.7136 ± 0.0429,0.4100 ± 0.0334,0.6866 ± 0.0461,0.4006 ± 0.0345,0.6968 ± 0.0471,0.4220 ± 0.0372,0.7070 ± 0.0453,0.4200 ± 0.0438,0.7453 ± 0.0358,0.4100 ± 0.0334,0.6866 ± 0.0461,0.6832 ± 0.0149,0.8298 ± 0.0266,0.5290 ± 0.0275,0.7547 ± 0.0375
1,Gradient Boosting Classifier SFS,"forward selection (n_features=10), StandardScaler",0.4204 ± 0.0361,0.6909 ± 0.0612,0.3904 ± 0.0339,0.6613 ± 0.0649,0.3795 ± 0.0377,0.6686 ± 0.0687,0.4022 ± 0.0341,0.6817 ± 0.0656,0.3890 ± 0.0503,0.7200 ± 0.0528,0.3904 ± 0.0339,0.6613 ± 0.0649,0.6715 ± 0.0176,0.8176 ± 0.0373,0.5118 ± 0.0289,0.7350 ± 0.0532
2,Gradient Boosting Classifier SFS,"forward selection (n_features=10), MinMaxScaler",0.4465 ± 0.0310,0.7088 ± 0.0415,0.4187 ± 0.0215,0.6819 ± 0.0433,0.4128 ± 0.0195,0.6918 ± 0.0435,0.4323 ± 0.0248,0.7022 ± 0.0431,0.4305 ± 0.0268,0.7383 ± 0.0311,0.4187 ± 0.0215,0.6819 ± 0.0433,0.6877 ± 0.0118,0.8274 ± 0.0267,0.5365 ± 0.0183,0.7510 ± 0.0360


#### XGBoost

In [29]:
y_train_xgb = y_train.replace({0:0, 1:1, -1:2})

# Modelo base
XGB_classifier = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", XGB_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGB_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="XGB_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", XGB_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="XGB_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_xgb = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_xgb

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_Classifier,"Default, No Scaler (Pipeline)",0.4163 ± 0.0273,0.9461 ± 0.0053,0.3953 ± 0.0217,0.9412 ± 0.0052,0.3920 ± 0.0231,0.9452 ± 0.0048,0.4096 ± 0.0257,0.9460 ± 0.0052,0.3948 ± 0.0255,0.9505 ± 0.0044,0.3953 ± 0.0217,0.9412 ± 0.0052,0.6845 ± 0.0107,0.9686 ± 0.0033,0.5201 ± 0.0182,0.9548 ± 0.0042
1,XGB_Classifier,"Default, StandardScaler (Pipeline)",0.4163 ± 0.0273,0.9461 ± 0.0053,0.3953 ± 0.0217,0.9412 ± 0.0052,0.3920 ± 0.0231,0.9452 ± 0.0048,0.4096 ± 0.0257,0.9460 ± 0.0052,0.3948 ± 0.0255,0.9505 ± 0.0044,0.3953 ± 0.0217,0.9412 ± 0.0052,0.6845 ± 0.0107,0.9686 ± 0.0033,0.5201 ± 0.0182,0.9548 ± 0.0042
2,XGB_Classifier,"Default, MinMaxScaler (Pipeline)",0.4163 ± 0.0273,0.9461 ± 0.0053,0.3953 ± 0.0217,0.9412 ± 0.0052,0.3920 ± 0.0231,0.9452 ± 0.0048,0.4096 ± 0.0257,0.9460 ± 0.0052,0.3948 ± 0.0255,0.9505 ± 0.0044,0.3953 ± 0.0217,0.9412 ± 0.0052,0.6845 ± 0.0107,0.9686 ± 0.0033,0.5201 ± 0.0182,0.9548 ± 0.0042


### XGBoost SFS_Forward

In [30]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(XGB_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(XGB_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train_xgb,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="XGBoost Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(XGB_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train_xgb,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="XGBoost Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(XGB_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train_xgb,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="XGBoost Classifier SFS",
    parameters=f"forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_xgboost_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_xgboost_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGBoost Classifier SFS,forward selection (n_features=10),0.3887 ± 0.0245,0.8760 ± 0.0389,0.3682 ± 0.0177,0.8661 ± 0.0423,0.3643 ± 0.0204,0.8728 ± 0.0406,0.3816 ± 0.0256,0.8754 ± 0.0393,0.3691 ± 0.0224,0.8844 ± 0.0361,0.3682 ± 0.0177,0.8661 ± 0.0423,0.6688 ± 0.0139,0.9289 ± 0.0227,0.4961 ± 0.0165,0.8969 ± 0.0329
1,XGBoost Classifier SFS,"forward selection (n_features=10), StandardScaler",0.3887 ± 0.0245,0.8760 ± 0.0389,0.3682 ± 0.0177,0.8661 ± 0.0423,0.3643 ± 0.0204,0.8728 ± 0.0406,0.3816 ± 0.0256,0.8754 ± 0.0393,0.3691 ± 0.0224,0.8844 ± 0.0361,0.3682 ± 0.0177,0.8661 ± 0.0423,0.6688 ± 0.0139,0.9289 ± 0.0227,0.4961 ± 0.0165,0.8969 ± 0.0329
2,XGBoost Classifier SFS,"forward selection (n_features=10), MinMaxScaler",0.3887 ± 0.0245,0.8760 ± 0.0389,0.3682 ± 0.0177,0.8661 ± 0.0423,0.3643 ± 0.0204,0.8728 ± 0.0406,0.3816 ± 0.0256,0.8754 ± 0.0393,0.3691 ± 0.0224,0.8844 ± 0.0361,0.3682 ± 0.0177,0.8661 ± 0.0423,0.6688 ± 0.0139,0.9289 ± 0.0227,0.4961 ± 0.0165,0.8969 ± 0.0329


## Distance Model 

### KNN

In [31]:

# Modelo base
KNN_classifier = KNeighborsClassifier(n_neighbors=31)


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", KNN_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="KNeighbors_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNN_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="KNeighbors_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", KNN_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="KNeighbors_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_knn = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_knn

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNeighbors_Classifier,"Default, No Scaler (Pipeline)",0.4505 ± 0.0269,0.5010 ± 0.0111,0.4041 ± 0.0272,0.4565 ± 0.0097,0.3697 ± 0.0352,0.4342 ± 0.0124,0.4067 ± 0.0327,0.4662 ± 0.0114,0.3945 ± 0.0608,0.4939 ± 0.0213,0.4041 ± 0.0272,0.4565 ± 0.0097,0.6761 ± 0.0169,0.7045 ± 0.0048,0.5226 ± 0.0240,0.5671 ± 0.0079
1,KNeighbors_Classifier,"Default, StandardScaler (Pipeline)",0.4162 ± 0.0201,0.4931 ± 0.0218,0.3699 ± 0.0165,0.4416 ± 0.0209,0.3295 ± 0.0142,0.4029 ± 0.0233,0.3666 ± 0.0161,0.4431 ± 0.0227,0.3671 ± 0.0565,0.4638 ± 0.0459,0.3699 ± 0.0165,0.4416 ± 0.0209,0.6484 ± 0.0120,0.6955 ± 0.0125,0.4897 ± 0.0152,0.5541 ± 0.0180
2,KNeighbors_Classifier,"Default, MinMaxScaler (Pipeline)",0.4354 ± 0.0354,0.4876 ± 0.0165,0.3889 ± 0.0277,0.4373 ± 0.0152,0.3508 ± 0.0233,0.4016 ± 0.0146,0.3871 ± 0.0286,0.4400 ± 0.0153,0.4099 ± 0.1142,0.4720 ± 0.0211,0.3889 ± 0.0277,0.4373 ± 0.0152,0.6612 ± 0.0191,0.6919 ± 0.0103,0.5070 ± 0.0250,0.5500 ± 0.0136


### KNN SFS_Forward

In [32]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# Forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(KNN_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(KNN_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="KNN Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(KNN_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="KNN Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(KNN_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="KNN Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_knn_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_knn_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN Classifier SFS,Forward selection (n_features=10),0.4657 ± 0.0185,0.5199 ± 0.0103,0.4229 ± 0.0185,0.4766 ± 0.0159,0.3999 ± 0.0246,0.4565 ± 0.0301,0.4325 ± 0.0222,0.4872 ± 0.0230,0.4307 ± 0.0231,0.5111 ± 0.0318,0.4229 ± 0.0185,0.4766 ± 0.0159,0.6880 ± 0.0163,0.7167 ± 0.0093,0.5394 ± 0.0181,0.5844 ± 0.0135
1,KNN Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.4506 ± 0.0310,0.5110 ± 0.0081,0.4082 ± 0.0342,0.4624 ± 0.0082,0.3793 ± 0.0460,0.4323 ± 0.0173,0.4117 ± 0.0413,0.4688 ± 0.0120,0.4721 ± 0.0815,0.5008 ± 0.0273,0.4082 ± 0.0342,0.4624 ± 0.0082,0.6750 ± 0.0259,0.7096 ± 0.0059,0.5248 ± 0.0319,0.5728 ± 0.0074
2,KNN Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4382 ± 0.0331,0.4986 ± 0.0175,0.3955 ± 0.0324,0.4520 ± 0.0189,0.3705 ± 0.0387,0.4267 ± 0.0256,0.4022 ± 0.0372,0.4609 ± 0.0207,0.4592 ± 0.0973,0.4707 ± 0.0263,0.3955 ± 0.0324,0.4520 ± 0.0189,0.6686 ± 0.0250,0.7037 ± 0.0114,0.5141 ± 0.0302,0.5639 ± 0.0164


## Margin-Based Models  

### SVM (RBF kernel)


In [33]:

# Modelo base
SVM_RBF_classifier = SVC(kernel='rbf')


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", SVM_RBF_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVM_RBF_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SVM_RBF_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", SVM_RBF_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SVM_RBF_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_svm_rbf = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_svm_rbf

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_Classifier,"Default, No Scaler (Pipeline)",0.4725 ± 0.0286,0.4794 ± 0.0070,0.4154 ± 0.0246,0.4216 ± 0.0063,0.3567 ± 0.0221,0.3629 ± 0.0057,0.4036 ± 0.0248,0.4104 ± 0.0063,0.3196 ± 0.0234,0.3235 ± 0.0056,0.4154 ± 0.0246,0.4216 ± 0.0063,0.6737 ± 0.0154,0.6779 ± 0.0042,0.5289 ± 0.0217,0.5346 ± 0.0056
1,SVM_RBF_Classifier,"Default, StandardScaler (Pipeline)",0.4395 ± 0.0374,0.5546 ± 0.0107,0.3900 ± 0.0318,0.5038 ± 0.0115,0.3477 ± 0.0288,0.4827 ± 0.0146,0.3864 ± 0.0321,0.5150 ± 0.0134,0.3874 ± 0.0734,0.6517 ± 0.0099,0.3900 ± 0.0318,0.5038 ± 0.0115,0.6606 ± 0.0199,0.7249 ± 0.0076,0.5074 ± 0.0281,0.6043 ± 0.0101
2,SVM_RBF_Classifier,"Default, MinMaxScaler (Pipeline)",0.4395 ± 0.0282,0.5175 ± 0.0119,0.3868 ± 0.0227,0.4627 ± 0.0139,0.3372 ± 0.0192,0.4291 ± 0.0217,0.3768 ± 0.0244,0.4659 ± 0.0188,0.3657 ± 0.0318,0.6071 ± 0.0418,0.3868 ± 0.0227,0.4627 ± 0.0139,0.6566 ± 0.0120,0.6991 ± 0.0101,0.5038 ± 0.0191,0.5687 ± 0.0126


### SVM (RBF kernel) SFS_Forward

In [34]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(SVM_RBF_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(SVM_RBF_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="SVM RBF Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SVM_RBF_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="SVM RBF Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SVM_RBF_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="SVM RBF Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_svm_rbf_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_svm_rbf_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM RBF Classifier SFS,Forward selection (n_features=10),0.4574 ± 0.0219,0.4839 ± 0.0097,0.4022 ± 0.0191,0.4258 ± 0.0094,0.3456 ± 0.0169,0.3665 ± 0.0085,0.3911 ± 0.0190,0.4144 ± 0.0092,0.3081 ± 0.0155,0.3265 ± 0.0067,0.4022 ± 0.0191,0.4258 ± 0.0094,0.6651 ± 0.0125,0.6808 ± 0.0068,0.5172 ± 0.0171,0.5384 ± 0.0086
1,SVM RBF Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.4381 ± 0.0409,0.5385 ± 0.0171,0.3905 ± 0.0381,0.4891 ± 0.0178,0.3541 ± 0.0392,0.4663 ± 0.0242,0.3921 ± 0.0401,0.4989 ± 0.0209,0.4195 ± 0.1281,0.6224 ± 0.0346,0.3905 ± 0.0381,0.4891 ± 0.0178,0.6655 ± 0.0265,0.7171 ± 0.0100,0.5095 ± 0.0350,0.5922 ± 0.0149
2,SVM RBF Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4340 ± 0.0256,0.4990 ± 0.0074,0.3855 ± 0.0223,0.4471 ± 0.0096,0.3458 ± 0.0275,0.4123 ± 0.0216,0.3834 ± 0.0261,0.4502 ± 0.0162,0.3575 ± 0.0560,0.5525 ± 0.0463,0.3855 ± 0.0223,0.4471 ± 0.0096,0.6591 ± 0.0185,0.6934 ± 0.0086,0.5040 ± 0.0215,0.5568 ± 0.0094


### SVM (Polynomial kernel) 

In [35]:

# Modelo base
SVM_Poly_classifier = SVC(kernel='poly')


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", SVM_Poly_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVM_Poly_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SVM_Poly_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", SVM_Poly_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SVM_Poly_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_svm_poly = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_svm_poly

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_Classifier,"Default, No Scaler (Pipeline)",0.4711 ± 0.0314,0.4763 ± 0.0110,0.4135 ± 0.0268,0.4177 ± 0.0101,0.3554 ± 0.0238,0.3588 ± 0.0094,0.4020 ± 0.0268,0.4061 ± 0.0104,0.3231 ± 0.0266,0.3249 ± 0.0073,0.4135 ± 0.0268,0.4177 ± 0.0101,0.6708 ± 0.0169,0.6736 ± 0.0075,0.5265 ± 0.0237,0.5304 ± 0.0094
1,SVM_Poly_Classifier,"Default, StandardScaler (Pipeline)",0.4395 ± 0.0162,0.5457 ± 0.0061,0.3851 ± 0.0131,0.4927 ± 0.0059,0.3303 ± 0.0164,0.4696 ± 0.0081,0.3661 ± 0.0194,0.4959 ± 0.0080,0.4093 ± 0.0266,0.7170 ± 0.0173,0.3851 ± 0.0131,0.4927 ± 0.0059,0.6534 ± 0.0061,0.7079 ± 0.0031,0.5015 ± 0.0108,0.5906 ± 0.0048
2,SVM_Poly_Classifier,"Default, MinMaxScaler (Pipeline)",0.4271 ± 0.0384,0.5336 ± 0.0086,0.3806 ± 0.0347,0.4840 ± 0.0087,0.3432 ± 0.0407,0.4663 ± 0.0107,0.3761 ± 0.0416,0.4949 ± 0.0106,0.3917 ± 0.0744,0.6276 ± 0.0324,0.3806 ± 0.0347,0.4840 ± 0.0087,0.6529 ± 0.0202,0.7085 ± 0.0063,0.4982 ± 0.0302,0.5856 ± 0.0077


### SVM (Polynomial kernel) SFS_Forward

In [42]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(SVM_Poly_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(SVM_Poly_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="SVM Poly Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SVM_Poly_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="SVM Poly Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SVM_Poly_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="SVM Poly Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_svm_poly_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_svm_poly_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM Poly Classifier SFS,Forward selection (n_features=10),0.4725 ± 0.0356,0.4791 ± 0.0095,0.4153 ± 0.0311,0.4203 ± 0.0089,0.3581 ± 0.0278,0.3613 ± 0.0081,0.4041 ± 0.0310,0.4086 ± 0.0091,0.3548 ± 0.0746,0.3475 ± 0.0470,0.4153 ± 0.0311,0.4203 ± 0.0089,0.6728 ± 0.0200,0.6754 ± 0.0072,0.5285 ± 0.0276,0.5328 ± 0.0084
1,SVM Poly Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.4546 ± 0.0192,0.5275 ± 0.0100,0.4004 ± 0.0201,0.4741 ± 0.0125,0.3508 ± 0.0381,0.4452 ± 0.0193,0.3861 ± 0.0335,0.4733 ± 0.0169,0.4572 ± 0.0900,0.6709 ± 0.0111,0.4004 ± 0.0201,0.4741 ± 0.0125,0.6596 ± 0.0098,0.6979 ± 0.0085,0.5138 ± 0.0167,0.5752 ± 0.0111
2,SVM Poly Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4409 ± 0.0451,0.5158 ± 0.0100,0.3913 ± 0.0420,0.4640 ± 0.0093,0.3526 ± 0.0479,0.4374 ± 0.0084,0.3894 ± 0.0470,0.4708 ± 0.0094,0.3789 ± 0.0909,0.5844 ± 0.0226,0.3913 ± 0.0420,0.4640 ± 0.0093,0.6604 ± 0.0248,0.6993 ± 0.0071,0.5080 ± 0.0368,0.5696 ± 0.0085


### SVM (Sigmoid kernel)  

In [43]:

# Modelo base
SVM_Sigmoid_classifier = SVC(kernel='sigmoid')


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", SVM_Sigmoid_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVM_Sigmoid_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="SVM_Sigmoid_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", SVM_Sigmoid_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="SVM_Sigmoid_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_svm_sigmoid = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_svm_sigmoid

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_Classifier,"Default, No Scaler (Pipeline)",0.4698 ± 0.0306,0.4602 ± 0.0174,0.4091 ± 0.0263,0.4004 ± 0.0162,0.3456 ± 0.0223,0.3367 ± 0.0171,0.3928 ± 0.0255,0.3830 ± 0.0189,0.3220 ± 0.0221,0.3121 ± 0.0111,0.4091 ± 0.0263,0.4004 ± 0.0162,0.6669 ± 0.0186,0.6611 ± 0.0122,0.5222 ± 0.0241,0.5144 ± 0.0152
1,SVM_Sigmoid_Classifier,"Default, StandardScaler (Pipeline)",0.4258 ± 0.0529,0.4251 ± 0.0100,0.3910 ± 0.0436,0.3892 ± 0.0066,0.3712 ± 0.0388,0.3710 ± 0.0119,0.3999 ± 0.0429,0.4003 ± 0.0080,0.3784 ± 0.0425,0.3769 ± 0.0095,0.3910 ± 0.0436,0.3892 ± 0.0066,0.6764 ± 0.0263,0.6758 ± 0.0090,0.5139 ± 0.0382,0.5128 ± 0.0068
2,SVM_Sigmoid_Classifier,"Default, MinMaxScaler (Pipeline)",0.4382 ± 0.0315,0.4485 ± 0.0155,0.3808 ± 0.0268,0.3899 ± 0.0135,0.3198 ± 0.0257,0.3293 ± 0.0127,0.3629 ± 0.0293,0.3728 ± 0.0134,0.3353 ± 0.0511,0.3553 ± 0.0402,0.3808 ± 0.0268,0.3899 ± 0.0135,0.6486 ± 0.0192,0.6537 ± 0.0091,0.4968 ± 0.0247,0.5048 ± 0.0122


### SVM (Sigmoid kernel) SFS_Forward

In [44]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(SVM_Sigmoid_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(SVM_Sigmoid_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="SVM Sigmoid Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SVM_Sigmoid_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="SVM Sigmoid Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(SVM_Sigmoid_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="SVM Sigmoid Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_svm_sigmoid_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_svm_sigmoid_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM Sigmoid Classifier SFS,Forward selection (n_features=10),0.3929 ± 0.0529,0.4107 ± 0.0305,0.3822 ± 0.0514,0.4024 ± 0.0106,0.3598 ± 0.0538,0.3777 ± 0.0135,0.3755 ± 0.0521,0.3923 ± 0.0098,0.3620 ± 0.0672,0.3891 ± 0.0388,0.3822 ± 0.0514,0.4024 ± 0.0106,0.6965 ± 0.0300,0.7085 ± 0.0164,0.5152 ± 0.0438,0.5338 ± 0.0073
1,SVM Sigmoid Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.4244 ± 0.0267,0.4464 ± 0.0212,0.3890 ± 0.0284,0.4135 ± 0.0171,0.3667 ± 0.0402,0.3998 ± 0.0184,0.3969 ± 0.0350,0.4256 ± 0.0171,0.3595 ± 0.0513,0.4171 ± 0.0212,0.3890 ± 0.0284,0.4135 ± 0.0171,0.6766 ± 0.0234,0.6884 ± 0.0119,0.5129 ± 0.0271,0.5335 ± 0.0151
2,SVM Sigmoid Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4450 ± 0.0349,0.4547 ± 0.0135,0.4034 ± 0.0303,0.4099 ± 0.0077,0.3709 ± 0.0326,0.3710 ± 0.0144,0.4042 ± 0.0333,0.4083 ± 0.0079,0.4734 ± 0.0870,0.3880 ± 0.0380,0.4034 ± 0.0303,0.4099 ± 0.0077,0.6749 ± 0.0205,0.6803 ± 0.0036,0.5216 ± 0.0272,0.5281 ± 0.0063


## Probabilistic Models  

### Multinomial Naive Bayes  


In [45]:

# Modelo base
MNB_classifier = MultinomialNB()


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", MNB_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="MultinomialNB_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)



# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", MNB_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_mm,
    modelo_name="MultinomialNB_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_mnb = pd.concat([df_1, df_2], ignore_index=True)
final_results_mnb

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MultinomialNB_Classifier,"Default, No Scaler (Pipeline)",0.4533 ± 0.0269,0.4938 ± 0.0140,0.4149 ± 0.0251,0.4586 ± 0.0125,0.4022 ± 0.0268,0.4521 ± 0.0129,0.4317 ± 0.0268,0.4764 ± 0.0132,0.4148 ± 0.0379,0.4734 ± 0.0138,0.4149 ± 0.0251,0.4586 ± 0.0125,0.6902 ± 0.0153,0.7098 ± 0.0076,0.5350 ± 0.0215,0.5705 ± 0.0108
1,MultinomialNB_Classifier,"Default, MinMaxScaler (Pipeline)",0.4395 ± 0.0146,0.4550 ± 0.0061,0.3795 ± 0.0127,0.3967 ± 0.0077,0.3041 ± 0.0188,0.3309 ± 0.0178,0.3443 ± 0.0187,0.3677 ± 0.0122,0.3854 ± 0.0837,0.4884 ± 0.0551,0.3795 ± 0.0127,0.3967 ± 0.0077,0.6460 ± 0.0079,0.6539 ± 0.0047,0.4951 ± 0.0107,0.5093 ± 0.0066


### Multinomial Naive Bayes  SFS_Forward

In [48]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(MNB_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(MNB_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="MNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)



# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(MNB_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="MNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_mnb_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_3_sfs],
    ignore_index=True
)

final_results_mnb_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MNB Classifier SFS,Forward selection (n_features=10),0.4615 ± 0.0367,0.4914 ± 0.0141,0.4240 ± 0.0355,0.4544 ± 0.0125,0.4127 ± 0.0388,0.4457 ± 0.0132,0.4404 ± 0.0373,0.4711 ± 0.0130,0.4311 ± 0.0540,0.4726 ± 0.0174,0.4240 ± 0.0355,0.4544 ± 0.0125,0.6916 ± 0.0177,0.7059 ± 0.0060,0.5412 ± 0.0289,0.5663 ± 0.0102
1,MNB Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4423 ± 0.0127,0.4502 ± 0.0049,0.3805 ± 0.0107,0.3891 ± 0.0055,0.3003 ± 0.0186,0.3120 ± 0.0178,0.3422 ± 0.0193,0.3518 ± 0.0131,0.3768 ± 0.0270,0.4816 ± 0.1257,0.3805 ± 0.0107,0.3891 ± 0.0055,0.6444 ± 0.0085,0.6470 ± 0.0047,0.4951 ± 0.0097,0.5017 ± 0.0052


### Bernoulli Naive Bayes  


In [49]:

# Modelo base
BNB_classifier = BernoulliNB()


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", BNB_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="BernoulliNB_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", BNB_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="BernoulliNB_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", BNB_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="BernoulliNB_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_bernoulli_nb = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_bernoulli_nb

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BernoulliNB_Classifier,"Default, No Scaler (Pipeline)",0.4135 ± 0.0207,0.4536 ± 0.0031,0.3637 ± 0.0205,0.4048 ± 0.0049,0.3153 ± 0.0243,0.3620 ± 0.0117,0.3478 ± 0.0241,0.3916 ± 0.0062,0.3966 ± 0.0382,0.4686 ± 0.0204,0.3637 ± 0.0205,0.4048 ± 0.0049,0.6445 ± 0.0098,0.6660 ± 0.0062,0.4840 ± 0.0162,0.5192 ± 0.0053
1,BernoulliNB_Classifier,"Default, StandardScaler (Pipeline)",0.4698 ± 0.0292,0.4907 ± 0.0139,0.4252 ± 0.0275,0.4489 ± 0.0139,0.4032 ± 0.0302,0.4333 ± 0.0164,0.4366 ± 0.0284,0.4621 ± 0.0152,0.4366 ± 0.0538,0.4760 ± 0.0142,0.4252 ± 0.0275,0.4489 ± 0.0139,0.6899 ± 0.0112,0.6998 ± 0.0093,0.5415 ± 0.0215,0.5605 ± 0.0124
2,BernoulliNB_Classifier,"Default, MinMaxScaler (Pipeline)",0.4121 ± 0.0210,0.4550 ± 0.0043,0.3635 ± 0.0206,0.4068 ± 0.0072,0.3179 ± 0.0253,0.3673 ± 0.0180,0.3497 ± 0.0251,0.3969 ± 0.0135,0.3975 ± 0.0372,0.4672 ± 0.0209,0.3635 ± 0.0206,0.4068 ± 0.0072,0.6448 ± 0.0097,0.6677 ± 0.0077,0.4840 ± 0.0164,0.5212 ± 0.0074


### Bernoulli Naive Bayes  SFS_Forward

In [50]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(BNB_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(BNB_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="BNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(BNB_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="BNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(BNB_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="BNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_bnb_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_bnb_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB Classifier SFS,Forward selection (n_features=10),0.4190 ± 0.0264,0.4543 ± 0.0039,0.3636 ± 0.0247,0.4009 ± 0.0094,0.2993 ± 0.0323,0.3453 ± 0.0251,0.3369 ± 0.0298,0.3783 ± 0.0169,0.3646 ± 0.0717,0.4555 ± 0.0681,0.3636 ± 0.0247,0.4009 ± 0.0094,0.6414 ± 0.0115,0.6599 ± 0.0101,0.4827 ± 0.0195,0.5144 ± 0.0098
1,BNB Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.4904 ± 0.0249,0.5024 ± 0.0118,0.4433 ± 0.0226,0.4570 ± 0.0129,0.4176 ± 0.0236,0.4350 ± 0.0172,0.4519 ± 0.0244,0.4670 ± 0.0146,0.4865 ± 0.0349,0.5037 ± 0.0248,0.4433 ± 0.0226,0.4570 ± 0.0129,0.6953 ± 0.0131,0.7027 ± 0.0081,0.5551 ± 0.0193,0.5666 ± 0.0112
2,BNB Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4203 ± 0.0243,0.4550 ± 0.0045,0.3655 ± 0.0219,0.4017 ± 0.0094,0.3018 ± 0.0307,0.3463 ± 0.0249,0.3388 ± 0.0284,0.3793 ± 0.0165,0.3720 ± 0.0689,0.4573 ± 0.0689,0.3655 ± 0.0219,0.4017 ± 0.0094,0.6420 ± 0.0116,0.6605 ± 0.0101,0.4843 ± 0.0176,0.5151 ± 0.0098


### Gaussian Naive Bayes

In [51]:

# Modelo base
GNB_classifier = GaussianNB()


# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", GNB_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="GaussianNB_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GNB_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="GaussianNB_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", GNB_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="GaussianNB_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_gaussian_nb = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_gaussian_nb

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GaussianNB_Classifier,"Default, No Scaler (Pipeline)",0.4368 ± 0.0142,0.4677 ± 0.0087,0.3909 ± 0.0130,0.4207 ± 0.0126,0.3543 ± 0.0278,0.3906 ± 0.0275,0.3840 ± 0.0245,0.4198 ± 0.0254,0.4237 ± 0.0304,0.4760 ± 0.0118,0.3909 ± 0.0130,0.4207 ± 0.0126,0.6630 ± 0.0104,0.6784 ± 0.0120,0.5091 ± 0.0122,0.5342 ± 0.0127
1,GaussianNB_Classifier,"Default, StandardScaler (Pipeline)",0.4272 ± 0.0259,0.4626 ± 0.0176,0.3820 ± 0.0290,0.4155 ± 0.0222,0.3392 ± 0.0578,0.3796 ± 0.0488,0.3676 ± 0.0561,0.4081 ± 0.0478,0.4040 ± 0.0632,0.4793 ± 0.0172,0.3820 ± 0.0290,0.4155 ± 0.0222,0.6564 ± 0.0226,0.6742 ± 0.0199,0.5006 ± 0.0277,0.5292 ± 0.0220
2,GaussianNB_Classifier,"Default, MinMaxScaler (Pipeline)",0.4327 ± 0.0177,0.4657 ± 0.0120,0.3871 ± 0.0194,0.4186 ± 0.0165,0.3483 ± 0.0397,0.3865 ± 0.0353,0.3775 ± 0.0369,0.4156 ± 0.0333,0.4192 ± 0.0368,0.4771 ± 0.0134,0.3871 ± 0.0194,0.4186 ± 0.0165,0.6601 ± 0.0155,0.6766 ± 0.0154,0.5054 ± 0.0185,0.5322 ± 0.0165


### Gaussian Naive Bayes SFS_Forward

In [52]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(GNB_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(GNB_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="GNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(GNB_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="GNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(GNB_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="GNB Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_gnb_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_gnb_classifier_sfs_forward

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB Classifier SFS,Forward selection (n_features=10),0.4341 ± 0.0135,0.4794 ± 0.0109,0.3920 ± 0.0139,0.4388 ± 0.0113,0.3713 ± 0.0159,0.4245 ± 0.0147,0.4021 ± 0.0117,0.4514 ± 0.0144,0.4002 ± 0.0324,0.4645 ± 0.0174,0.3920 ± 0.0139,0.4388 ± 0.0113,0.6733 ± 0.0044,0.6949 ± 0.0079,0.5137 ± 0.0105,0.5522 ± 0.0101
1,GNB Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.4341 ± 0.0135,0.4794 ± 0.0109,0.3920 ± 0.0139,0.4388 ± 0.0113,0.3713 ± 0.0159,0.4245 ± 0.0147,0.4021 ± 0.0117,0.4514 ± 0.0144,0.4002 ± 0.0324,0.4645 ± 0.0174,0.3920 ± 0.0139,0.4388 ± 0.0113,0.6733 ± 0.0044,0.6949 ± 0.0079,0.5137 ± 0.0105,0.5522 ± 0.0101
2,GNB Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4341 ± 0.0135,0.4794 ± 0.0109,0.3920 ± 0.0139,0.4388 ± 0.0113,0.3713 ± 0.0159,0.4245 ± 0.0147,0.4021 ± 0.0117,0.4514 ± 0.0144,0.4002 ± 0.0324,0.4645 ± 0.0174,0.3920 ± 0.0139,0.4388 ± 0.0113,0.6733 ± 0.0044,0.6949 ± 0.0079,0.5137 ± 0.0105,0.5522 ± 0.0101


# Neural Networks

### MLP

In [53]:


# Modelo base
MLP_classifier = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    max_iter=500,
    random_state=42
)

    

# 1) NO SCALER 
pipe_none = Pipeline([
    ("clf", MLP_classifier)
])

results_none = cross_validate(
    pipe_none,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_Classifier",
    parameters="Default, No Scaler (Pipeline)",
    sep=" ± "
)

# 2) StandardScaler 
pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLP_classifier)
])

results_ss = cross_validate(
    pipe_ss,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_2 = cv_results_to_row(
    results_ss,
    modelo_name="MLP_Classifier",
    parameters="Default, StandardScaler (Pipeline)",
    sep=" ± "
)

# 3) MinMaxScaler 
pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("clf", MLP_classifier)
])

results_mm = cross_validate(
    pipe_mm,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df_3 = cv_results_to_row(
    results_mm,
    modelo_name="MLP_Classifier",
    parameters="Default, MinMaxScaler (Pipeline)",
    sep=" ± "
)

final_results_mlp = pd.concat([df_1, df_2, df_3], ignore_index=True)
final_results_mlp

/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't conv

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_Classifier,"Default, No Scaler (Pipeline)",0.4574 ± 0.0232,0.5361 ± 0.0181,0.4165 ± 0.0253,0.4947 ± 0.0234,0.4002 ± 0.0329,0.4839 ± 0.0307,0.4284 ± 0.0284,0.5092 ± 0.0261,0.4405 ± 0.0550,0.5574 ± 0.0261,0.4165 ± 0.0253,0.4947 ± 0.0234,0.6786 ± 0.0127,0.7227 ± 0.0153,0.5315 ± 0.0210,0.5979 ± 0.0204
1,MLP_Classifier,"Default, StandardScaler (Pipeline)",0.3874 ± 0.0316,0.8901 ± 0.0044,0.3704 ± 0.0338,0.8828 ± 0.0038,0.3644 ± 0.0325,0.8887 ± 0.0037,0.3780 ± 0.0316,0.8899 ± 0.0042,0.3703 ± 0.0338,0.8979 ± 0.0042,0.3704 ± 0.0338,0.8828 ± 0.0038,0.6676 ± 0.0210,0.9369 ± 0.0026,0.4971 ± 0.0303,0.9094 ± 0.0031
2,MLP_Classifier,"Default, MinMaxScaler (Pipeline)",0.4025 ± 0.0240,0.6597 ± 0.0149,0.3807 ± 0.0224,0.6393 ± 0.0122,0.3770 ± 0.0240,0.6462 ± 0.0136,0.3919 ± 0.0232,0.6553 ± 0.0141,0.3913 ± 0.0335,0.6705 ± 0.0208,0.3807 ± 0.0224,0.6393 ± 0.0122,0.6655 ± 0.0116,0.8062 ± 0.0054,0.5032 ± 0.0183,0.7179 ± 0.0093


### MLP SFS_Forward

In [56]:

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


N_FEATURES = 10  


# forward Sequential Feature Selector (wrapper)
sfs_forward = SequentialFeatureSelector(
    estimator=clone(MLP_classifier),
    n_features_to_select=N_FEATURES,
    direction="forward",
    scoring="balanced_accuracy",
    cv=cv_inner,
    n_jobs=-1
)

# 1) No scaler + Forward SFS
pipe = Pipeline([
    ("sfs", sfs_forward),
    ("clf", clone(MLP_classifier)),
])

results = cross_validate(
    pipe, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_1_sfs = cv_results_to_row(
    results,
    modelo_name="MLP Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES})",
    sep=" ± "
)


# 2) StandardScaler + Forward SFS

pipe_ss = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(MLP_classifier)),
])

results_ss = cross_validate(
    pipe_ss, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_2_sfs = cv_results_to_row(
    results_ss,
    modelo_name="MLP Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), StandardScaler",
    sep=" ± "
)


# 3) MinMaxScaler + Forward SFS

pipe_mm = Pipeline([
    ("scaler", MinMaxScaler()),
    ("sfs", sfs_forward),
    ("clf", clone(MLP_classifier)),
])

results_mm = cross_validate(
    pipe_mm, X_train, y_train,
    cv=cv_outer,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

df_3_sfs = cv_results_to_row(
    results_mm,
    modelo_name="MLP Classifier SFS",
    parameters=f"Forward selection (n_features={N_FEATURES}), MinMaxScaler",
    sep=" ± "
)


# Final table
final_results_mlp_classifier_sfs_forward = pd.concat(
    [df_1_sfs, df_2_sfs, df_3_sfs],
    ignore_index=True
)

final_results_mlp_classifier_sfs_forward

/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/fsc/Desktop/PD_Project_UofL/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't conv

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP Classifier SFS,Forward selection (n_features=10),0.4368 ± 0.0395,0.5378 ± 0.0388,0.3940 ± 0.0340,0.4940 ± 0.0424,0.3584 ± 0.0318,0.4736 ± 0.0576,0.3903 ± 0.0349,0.5014 ± 0.0503,0.3877 ± 0.0555,0.5689 ± 0.0589,0.3940 ± 0.0340,0.4940 ± 0.0424,0.6655 ± 0.0242,0.7228 ± 0.0245,0.5118 ± 0.0311,0.5974 ± 0.0353
1,MLP Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.3956 ± 0.0136,0.7816 ± 0.0407,0.3743 ± 0.0154,0.7672 ± 0.0460,0.3710 ± 0.0146,0.7740 ± 0.0450,0.3883 ± 0.0101,0.7796 ± 0.0424,0.3787 ± 0.0188,0.7902 ± 0.0370,0.3743 ± 0.0154,0.7672 ± 0.0460,0.6717 ± 0.0077,0.8761 ± 0.0244,0.5013 ± 0.0111,0.8197 ± 0.0361
2,MLP Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4285 ± 0.0398,0.6020 ± 0.0104,0.4037 ± 0.0350,0.5742 ± 0.0116,0.3959 ± 0.0326,0.5771 ± 0.0130,0.4133 ± 0.0368,0.5923 ± 0.0113,0.4100 ± 0.0353,0.6117 ± 0.0184,0.4037 ± 0.0350,0.5742 ± 0.0116,0.6801 ± 0.0214,0.7721 ± 0.0057,0.5238 ± 0.0307,0.6658 ± 0.0091


# Model Default Unification DataFrames

In [ ]:
data_defalut_models = pd.concat([df_dummy,final_results,final_results_svc,final_results_sgd,
                                 final_results_tree,final_results_rf,final_results_et,
                                 final_results_gb,final_results_xgb,final_results_knn,
                                 final_results_svm_rbf,final_results_svm_poly,final_results_svm_sigmoid,
                                 final_results_mnb,final_results_bernoulli_nb,final_results_bernoulli_nb,
                                 final_results_mlp], ignore_index=True)
data_defalut_models.to_csv(ROOT/'DATA_MODEL_Dev/Motor_default_models_comparison.csv', index=False)
data_defalut_models.head()

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,DummyClassifier,"Most frequent (Baseline), No Scaler (Pipeline)",0.3956 ± 0.0020,0.3956 ± 0.0005,0.3333 ± 0.0000,0.3333 ± 0.0000,0.1890 ± 0.0007,0.1890 ± 0.0002,0.2243 ± 0.0020,0.2243 ± 0.0005,0.1319 ± 0.0007,0.1319 ± 0.0002,0.3333 ± 0.0000,0.3333 ± 0.0000,0.6044 ± 0.0020,0.6044 ± 0.0005,0.4489 ± 0.0008,0.4488 ± 0.0002
1,LogisticRegression,"Balanced, max_iter=1000, Default Model, No Sca...",0.4410 ± 0.0559,0.4828 ± 0.0190,0.4308 ± 0.0579,0.4787 ± 0.0195,0.4288 ± 0.0611,0.4754 ± 0.0196,0.4419 ± 0.0586,0.4842 ± 0.0194,0.4342 ± 0.0632,0.4756 ± 0.0194,0.4308 ± 0.0579,0.4787 ± 0.0195,0.7163 ± 0.0273,0.7377 ± 0.0097,0.5548 ± 0.0478,0.5942 ± 0.0160
2,LogisticRegression,"Balanced, max_iter=1000, Default Model, Standa...",0.4314 ± 0.0536,0.4780 ± 0.0177,0.4198 ± 0.0548,0.4742 ± 0.0180,0.4181 ± 0.0581,0.4707 ± 0.0181,0.4321 ± 0.0557,0.4793 ± 0.0180,0.4237 ± 0.0597,0.4710 ± 0.0180,0.4198 ± 0.0548,0.4742 ± 0.0180,0.7109 ± 0.0249,0.7354 ± 0.0093,0.5456 ± 0.0448,0.5905 ± 0.0149
3,LogisticRegression,"Balanced, max_iter=1000, Default Model, MinMax...",0.4533 ± 0.0563,0.4828 ± 0.0159,0.4369 ± 0.0577,0.4729 ± 0.0168,0.4355 ± 0.0610,0.4719 ± 0.0171,0.4520 ± 0.0589,0.4831 ± 0.0164,0.4393 ± 0.0637,0.4715 ± 0.0171,0.4369 ± 0.0577,0.4729 ± 0.0168,0.7163 ± 0.0287,0.7312 ± 0.0085,0.5588 ± 0.0483,0.5880 ± 0.0139
4,SVC_Linear,"Default Model, No Scaler (Pipeline)",0.4602 ± 0.0347,0.4924 ± 0.0121,0.4091 ± 0.0302,0.4401 ± 0.0145,0.3672 ± 0.0318,0.4006 ± 0.0250,0.4080 ± 0.0324,0.4407 ± 0.0206,0.3686 ± 0.0650,0.4920 ± 0.0086,0.4091 ± 0.0302,0.4401 ± 0.0145,0.6738 ± 0.0185,0.6906 ± 0.0109,0.5249 ± 0.0264,0.5513 ± 0.0135


In [38]:
sfs_models = pd.concat([final_results_logreg_sfs_forward,
                        final_results_linear_svc_sfs_forward,
                        final_results_sgd_classifier_sfs_forward,
                        final_results_tree_classifier_sfs_forward,
                        final_results_rf_classifier_sfs_forward,
                        final_results_extratree_classifier_sfs_forward,
                        final_results_gradient_boosting_classifier_sfs_forward,
                        final_results_xgboost_classifier_sfs_forward,
                        final_results_knn_classifier_sfs_forward,
                        final_results_svm_rbf_classifier_sfs_forward,
                        final_results_svm_poly_classifier_sfs_forward,
                        final_results_svm_sigmoid_classifier_sfs_forward,
                        final_results_mnb_classifier_sfs_forward,
                        final_results_bnb_classifier_sfs_forward,
                        final_results_gnb_classifier_sfs_forward,
                        final_results_mlp_classifier_sfs_forward], ignore_index=True)
sfs_models.to_csv(ROOT/'DATA_MODEL_Dev/Motor_sfs_models.csv')

NameError: name 'final_results_logreg_sfs_forward' is not defined

In [39]:
sfs_models=pd.read_csv(ROOT/'DATA_MODEL_Dev/Motor_sfs_models.csv')
sfs_models

,Unnamed: 0,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,0,LogReg SFS,Forward selection (n_features=10),0.4396 ± 0.0675,0.4784 ± 0.0226,0.4315 ± 0.0728,0.4733 ± 0.0214,0.4269 ± 0.0750,0.4694 ± 0.0221,0.4392 ± 0.0719,0.4792 ± 0.0223,0.4326 ± 0.0790,0.4708 ± 0.0212,0.4315 ± 0.0728,0.4733 ± 0.0214,0.7172 ± 0.0398,0.7367 ± 0.0094,0.5553 ± 0.0628,0.5904 ± 0.0170
1,1,LogReg SFS,"Forward selection (n_features=10), StandardScaler",0.4423 ± 0.0461,0.4839 ± 0.0208,0.4336 ± 0.0503,0.4800 ± 0.0210,0.4304 ± 0.0522,0.4760 ± 0.0212,0.4425 ± 0.0497,0.4853 ± 0.0208,0.4362 ± 0.0554,0.4774 ± 0.0206,0.4336 ± 0.0503,0.4800 ± 0.0210,0.7169 ± 0.0296,0.7406 ± 0.0100,0.5571 ± 0.0434,0.5961 ± 0.0170
2,2,LogReg SFS,"Forward selection (n_features=10), MinMaxScaler",0.4465 ± 0.0471,0.4873 ± 0.0176,0.4349 ± 0.0481,0.4777 ± 0.0186,0.4334 ± 0.0505,0.4758 ± 0.0188,0.4471 ± 0.0494,0.4871 ± 0.0178,0.4379 ± 0.0528,0.4761 ± 0.0189,0.4349 ± 0.0481,0.4777 ± 0.0186,0.7165 ± 0.0251,0.7349 ± 0.0088,0.5577 ± 0.0407,0.5925 ± 0.0150
3,3,LinearSVC SFS,Forward selection (n_features=10),0.4602 ± 0.0284,0.4856 ± 0.0091,0.4081 ± 0.0222,0.4333 ± 0.0105,0.3637 ± 0.0187,0.3943 ± 0.0195,0.4052 ± 0.0212,0.4340 ± 0.0154,0.3659 ± 0.0473,0.4813 ± 0.0234,0.4081 ± 0.0222,0.4333 ± 0.0105,0.6722 ± 0.0142,0.6855 ± 0.0078,0.5237 ± 0.0197,0.5450 ± 0.0097
4,4,LinearSVC SFS,"Forward selection (n_features=10), StandardScaler",0.4464 ± 0.0215,0.4876 ± 0.0089,0.3949 ± 0.0143,0.4341 ± 0.0129,0.3481 ± 0.0073,0.3916 ± 0.0249,0.3903 ± 0.0092,0.4327 ± 0.0200,0.3270 ± 0.0248,0.4992 ± 0.0343,0.3949 ± 0.0143,0.4341 ± 0.0129,0.6654 ± 0.0062,0.6859 ± 0.0106,0.5125 ± 0.0114,0.5456 ± 0.0123
5,5,LinearSVC SFS,"Forward selection (n_features=10), MinMaxScaler",0.4547 ± 0.0304,0.4900 ± 0.0143,0.4026 ± 0.0260,0.4353 ± 0.0179,0.3578 ± 0.0281,0.3895 ± 0.0327,0.3990 ± 0.0274,0.4320 ± 0.0265,0.3630 ± 0.0819,0.4424 ± 0.0960,0.4026 ± 0.0260,0.4353 ± 0.0179,0.6669 ± 0.0154,0.6862 ± 0.0127,0.5180 ± 0.0224,0.5465 ± 0.0163
6,6,SGD Classifier SFS,Forward selection (n_features=10),0.4574 ± 0.0373,0.4485 ± 0.0306,0.4214 ± 0.0390,0.4167 ± 0.0466,0.3583 ± 0.0746,0.3510 ± 0.0845,0.3869 ± 0.0690,0.3762 ± 0.0773,0.4106 ± 0.0613,0.4672 ± 0.0880,0.4214 ± 0.0390,0.4167 ± 0.0466,0.6887 ± 0.0414,0.6860 ± 0.0501,0.5386 ± 0.0409,0.5345 ± 0.0492
7,7,SGD Classifier SFS,"Forward selection (n_features=10), StandardScaler",0.3955 ± 0.0494,0.4015 ± 0.0406,0.3894 ± 0.0444,0.3946 ± 0.0258,0.3733 ± 0.0535,0.3843 ± 0.0281,0.3838 ± 0.0565,0.3958 ± 0.0363,0.3857 ± 0.0558,0.3998 ± 0.0271,0.3894 ± 0.0444,0.3946 ± 0.0258,0.6896 ± 0.0295,0.7000 ± 0.0082,0.5178 ± 0.0394,0.5253 ± 0.0179
8,8,SGD Classifier SFS,"Forward selection (n_features=10), MinMaxScaler",0.4574 ± 0.0267,0.4688 ± 0.0089,0.4199 ± 0.0279,0.4324 ± 0.0103,0.3743 ± 0.0454,0.3889 ± 0.0350,0.4055 ± 0.0430,0.4186 ± 0.0323,0.4361 ± 0.0619,0.4533 ± 0.0812,0.4199 ± 0.0279,0.4324 ± 0.0103,0.6916 ± 0.0208,0.6996 ± 0.0082,0.5388 ± 0.0260,0.5500 ± 0.0096
9,9,Tree Classifier SFS,forward selection (n_features=10),0.4437 ± 0.0309,0.5450 ± 0.0183,0.4142 ± 0.0282,0.5159 ± 0.0167,0.4059 ± 0.0307,0.5103 ± 0.0197,0.4281 ± 0.0327,0.5291 ± 0.0191,0.4308 ± 0.0288,0.5538 ± 0.0240,0.4142 ± 0.0282,0.5159 ± 0.0167,0.6873 ± 0.0233,0.7426 ± 0.0122,0.5335 ± 0.0265,0.6189 ± 0.0148
